<a href="https://colab.research.google.com/github/fernandodeeke/Hydraulics/blob/main/turbulent_2_tank_slide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact, FloatSlider, VBox, HBox, Label, HTML
import ipywidgets as widgets

# ==============================================================
# TWO-TANK SYSTEM -- INTERACTIVE NOTEBOOK
# Adjust sliders to explore how each physical parameter affects
# the system's transient response and equilibrium heights.
# ==============================================================

# --------------------------------------------------------------
# SLIDER DEFINITIONS
# Each slider description includes: symbol, unit, and meaning.
# Ranges are chosen to stay physically plausible.
# --------------------------------------------------------------

slider_style  = {'description_width': '260px'}
slider_layout = widgets.Layout(width='600px')

def make_slider(desc, val, lo, hi, step, fmt='.4g'):
    return FloatSlider(
        value=val, min=lo, max=hi, step=step,
        description=desc,
        readout_format=fmt,
        style=slider_style, layout=slider_layout,
        continuous_update=True
    )

def header(text):
    return HTML("<b style='font-size:13px; color:#444'>" + text + "</b>")


# -- Initial Conditions ----------------------------------------
w_h1_0 = make_slider('h1(0)  Initial level Tank 1 (m)', 0.00, 0.00, 0.40, 0.005, '.3f')
w_h2_0 = make_slider('h2(0)  Initial level Tank 2 (m)', 0.00, 0.00, 0.40, 0.005, '.3f')

# -- Tank Geometry ---------------------------------------------
w_A1 = make_slider('A1  Cross-section Tank 1 (m^2)',  0.02, 0.005, 0.10, 0.005, '.4f')
w_A2 = make_slider('A2  Cross-section Tank 2 (m^2)',  0.02, 0.005, 0.10, 0.005, '.4f')

# -- Inflow ----------------------------------------------------
# Slider uses L/min; converted to m^3/s inside plot().
# 1 L/min = 1/60 * 1e-3 m^3/s = 1.667e-5 m^3/s
w_q1_lpm = make_slider('q1  Inflow rate (L/min)', 1.00, 0.10, 5.00, 0.10, '.2f')

# -- Fluid Properties ------------------------------------------
# Slider uses mPa*s; converted to Pa*s inside plot().
w_mu  = make_slider('mu  Dynamic viscosity (mPa*s)', 1.00,   0.50, 5.00,   0.10, '.2f')
w_rho = make_slider('rho Fluid density (kg/m^3)',    1000.0, 800.0, 1200.0, 10.0, '.1f')

# -- Connecting Pipe (laminar, Hagen-Poiseuille) ---------------
# Slider uses mm; converted to m inside plot().
w_L = make_slider('L   Pipe length (m)',        0.20, 0.05, 1.00, 0.01, '.3f')
w_r = make_slider('r   Pipe radius (mm)',       1.50, 0.50, 5.00, 0.10, '.2f')

# -- Outlet Valve (turbulent, Torricelli) ----------------------
# Slider uses units of 1e-5 m^2; converted inside plot().
w_Cd = make_slider('Cd  Discharge coefficient (-)',    0.62, 0.30, 0.90, 0.01,  '.3f')
w_a  = make_slider('a   Valve area (x1e-5 m^2)',       1.50, 0.10, 5.00, 0.10,  '.2f')

# -- Simulation Horizon ----------------------------------------
w_T = make_slider('T   Simulation horizon (s)', 1500.0, 300.0, 3600.0, 60.0, '.0f')


# ==============================================================
# SIMULATION FUNCTION
# All arguments are in SI units.
# Returns the ODE solution and key derived quantities.
# ==============================================================

def run_simulation(h1_0, h2_0, A1, A2, q1, mu, rho, L, r, Cd, a, T):

    g = 9.81  # gravitational acceleration (m/s^2) -- physical constant, not a slider

    # -- Derived Coefficient 1: Hagen-Poiseuille Resistance ----
    # Models the laminar flow through the connecting pipe.
    # From: q12 = (pi * r^4 * rho * g * dh) / (8 * mu * L)
    # Rearranged to head form: q12 = (h1 - h2) / R12
    R12 = (8 * mu * L) / (rho * g * np.pi * r**4)

    # -- Derived Coefficient 2: Turbulent Valve Coefficient ----
    # Models the turbulent outflow through the sharp-edged orifice.
    # From Torricelli: q2 = Cd * a * sqrt(2 * g * h2)
    # Lumped form:     q2 = k2 * sqrt(h2)
    k2 = Cd * a * np.sqrt(2 * g)

    # -- Analytical Steady-State Calculation -------------------
    # At equilibrium dh/dt = 0 for both tanks, so:
    #   q1 = q2 = k2 * sqrt(h2*)  =>  h2* = (q1 / k2)^2
    #   q1 = (h1* - h2*) / R12   =>  h1* = R12 * q1 + h2*
    h2_star = (q1 / k2) ** 2
    h1_star = R12 * q1 + h2_star

    # -- ODE System (mass balance on each tank) ----------------
    def two_tanks(t, h):
        h1 = max(0.0, h[0])  # physical floor: height cannot go negative
        h2 = max(0.0, h[1])

        q12 = (h1 - h2) / R12    # laminar flow between tanks (linear in delta-h)
        q2  = k2 * np.sqrt(h2)   # turbulent outflow from Tank 2 (nonlinear in h2)

        dh1_dt = (q1  - q12) / A1  # mass balance Tank 1
        dh2_dt = (q12 - q2)  / A2  # mass balance Tank 2
        return [dh1_dt, dh2_dt]

    # -- Numerical Integration (Runge-Kutta 45) ----------------
    t_eval = np.linspace(0, T, 1000)
    sol = solve_ivp(
        two_tanks, (0, T), [h1_0, h2_0],
        t_eval=t_eval, method='RK45', max_step=T / 500
    )

    return sol, h1_star, h2_star, R12, k2


# ==============================================================
# PLOT FUNCTION
# Called automatically by ipywidgets on every slider change.
# Slider values are converted from display units to SI here.
# ==============================================================

def plot(h1_0, h2_0, A1, A2, q1_lpm, mu_mPas, rho, L, r_mm, Cd, a_1e5, T):

    # Unit conversions: slider display units -> SI
    q1 = q1_lpm  * 1e-3 / 60   # L/min   -> m^3/s
    mu = mu_mPas * 1e-3         # mPa*s   -> Pa*s
    r  = r_mm    * 1e-3         # mm      -> m
    a  = a_1e5   * 1e-5         # x1e-5   -> m^2

    sol, h1_star, h2_star, R12, k2 = run_simulation(
        h1_0, h2_0, A1, A2, q1, mu, rho, L, r, Cd, a, T
    )

    fig, ax = plt.subplots(figsize=(10, 5))

    # Transient responses -- convert m to cm for readability on the y-axis
    ax.plot(sol.t, sol.y[0] * 100, color='steelblue',  lw=2.0, label='Tank 1  h1(t)')
    ax.plot(sol.t, sol.y[1] * 100, color='darkorange', lw=2.0, label='Tank 2  h2(t)')

    # Equilibrium reference lines (dashed)
    ax.axhline(h1_star * 100, color='steelblue',  lw=1.2, ls='--', alpha=0.6,
               label='Steady state  h1* = {:.2f} cm'.format(h1_star * 100))
    ax.axhline(h2_star * 100, color='darkorange', lw=1.2, ls='--', alpha=0.6,
               label='Steady state  h2* = {:.2f} cm'.format(h2_star * 100))

    ax.set_xlabel('Time (s)',          fontsize=12)
    ax.set_ylabel('Water level (cm)',  fontsize=12)
    ax.set_title('Two-Tank System -- Transient Response', fontsize=13)
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, ls=':', alpha=0.5)
    ax.set_xlim(0, T)

    # Annotation: show derived quantities directly on the plot
    info = ('R12 = {:.2f} s/m^2     '
            'k2 = {:.2e} m^2.5/s     '
            'h1* = {:.2f} cm     '
            'h2* = {:.2f} cm').format(R12, k2, h1_star * 100, h2_star * 100)
    ax.text(0.02, 0.97, info, transform=ax.transAxes, fontsize=9,
            va='top', color='#333',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))

    plt.tight_layout()
    plt.show()


# ==============================================================
# WIDGET LAYOUT
# Sliders are grouped by physical category, matching the
# parameter block structure in the original notebook.
# ==============================================================

ui = VBox([
    header('-- Initial Conditions -------------------------------------------'),
    w_h1_0, w_h2_0,

    header('-- Tank Geometry ------------------------------------------------'),
    w_A1, w_A2,

    header('-- Inflow -------------------------------------------------------'),
    w_q1_lpm,

    header('-- Fluid Properties ---------------------------------------------'),
    w_mu, w_rho,

    header('-- Connecting Pipe  (Hagen-Poiseuille, laminar) -----------------'),
    w_L, w_r,

    header('-- Outlet Valve  (Torricelli, turbulent) ------------------------'),
    w_Cd, w_a,

    header('-- Simulation ---------------------------------------------------'),
    w_T,
])

out = widgets.interactive_output(plot, {
    'h1_0':    w_h1_0,
    'h2_0':    w_h2_0,
    'A1':      w_A1,
    'A2':      w_A2,
    'q1_lpm':  w_q1_lpm,
    'mu_mPas': w_mu,
    'rho':     w_rho,
    'L':       w_L,
    'r_mm':    w_r,
    'Cd':      w_Cd,
    'a_1e5':   w_a,
    'T':       w_T,
})

display(ui, out)

Output()